# Binary Sentiment Classification with mBERT

## Objective
Fine-tune pretrained **mBERT** (`bert-base-multilingual-cased`) for binary HealthPH+ sentiment classification.

**Label contract:**

- `1 = worried/concerned`
- `0 = confident/not worried`

This notebook uses only sentiment-labeled files from `data/training_data`:

- `training_1.csv`
- `training_2.csv`

`gold_standard.csv` is excluded because it contains disease labels but no `sentiment` target.

This notebook intentionally does **not** use the existing ELECTRA notebooks, ELECTRA weights, or ELECTRA artifacts. It loads mBERT from the local Hugging Face cache with `local_files_only=True`.

> Current data note: both training files currently have only `sentiment = 0`. The validation cell will stop before training until at least one `sentiment = 1` worried/concerned example is added.

In [ ]:
# Environment setup
import json
import random
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

print('Notebook Python executable:', sys.executable)

try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing dependency 'torch'. Use the project virtual environment or install torch, then restart the kernel."
    ) from exc

try:
    accelerate_version = version('accelerate')
except PackageNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing dependency 'accelerate>=1.1.0'. Install it in the notebook kernel environment, then restart."
    ) from exc

try:
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        set_seed,
    )
    from transformers.utils import is_accelerate_available
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing dependency 'transformers'. Use the project virtual environment or install transformers, then restart."
    ) from exc

if not is_accelerate_available('1.1.0'):
    raise RuntimeError(
        'Transformers cannot see accelerate>=1.1.0 in this kernel. Restart after installing/upgrading accelerate.'
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print('Torch runtime device:', DEVICE)
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')

In [ ]:
# Config
CWD = Path.cwd()
ROOT_DIR = next((p for p in [CWD, *CWD.parents] if (p / '.git').exists()), CWD)

TRAINING_FILES = [
    ROOT_DIR / 'data' / 'training_data' / 'training_1.csv',
    ROOT_DIR / 'data' / 'training_data' / 'training_2.csv',
]

BASE_MODEL_NAME = 'bert-base-multilingual-cased'
MODEL_KEY = 'mbert_finetuned'
LOCAL_FILES_ONLY = True

LABEL_NAMES = {
    0: 'confident_not_worried',
    1: 'worried_concerned',
}
ID2LABEL = {idx: label for idx, label in LABEL_NAMES.items()}
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}

TEST_SIZE = 0.10
VAL_SIZE = 0.10
MAX_LENGTH = 128

BATCH_SIZE = 8
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

ARTIFACT_DIR = ROOT_DIR / 'artifacts' / 'binary_sentiment' / MODEL_KEY
REPORT_DIR = ROOT_DIR / 'reports' / 'binary_sentiment' / 'mbert'
TRAINER_OUTPUT_DIR = ARTIFACT_DIR / 'trainer_output'
BEST_MODEL_DIR = ARTIFACT_DIR / 'best_model'

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'root_dir': str(ROOT_DIR),
    'training_files': [str(p) for p in TRAINING_FILES],
    'base_model_name': BASE_MODEL_NAME,
    'local_files_only': LOCAL_FILES_ONLY,
    'artifact_dir': str(ARTIFACT_DIR),
    'report_dir': str(REPORT_DIR),
    'label_contract': LABEL_NAMES,
    'device': str(DEVICE),
}, indent=2))

In [ ]:
# Data loading + schema normalization
TEXT_COLUMNS = ('post', 'cleaned_text')
LABEL_COLUMN = 'sentiment'


def load_sentiment_file(path):
    if not path.exists():
        raise FileNotFoundError(f'Missing training file: {path}')

    df = pd.read_csv(path)
    text_col = next((col for col in TEXT_COLUMNS if col in df.columns), None)
    if text_col is None:
        raise KeyError(f'{path.name} must contain one of these text columns: {TEXT_COLUMNS}')
    if LABEL_COLUMN not in df.columns:
        raise KeyError(f'{path.name} must contain `{LABEL_COLUMN}` for binary sentiment training')

    return pd.DataFrame({
        'text': df[text_col].astype(str),
        'label_raw': df[LABEL_COLUMN],
        'source_file': path.name,
    })


raw_df = pd.concat([load_sentiment_file(path) for path in TRAINING_FILES], ignore_index=True)
raw_df['text'] = raw_df['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
raw_df['label'] = pd.to_numeric(raw_df['label_raw'], errors='coerce')

clean_df = raw_df.dropna(subset=['label']).copy()
clean_df['label'] = clean_df['label'].astype(int)
clean_df = clean_df[clean_df['text'].str.len() > 0].copy()
clean_df = clean_df.drop_duplicates(subset=['text']).reset_index(drop=True)

print('Raw rows:', len(raw_df))
print('Clean rows:', len(clean_df))
print('\nRows by source:')
print(clean_df['source_file'].value_counts().to_string())
print('\nLabel counts:')
print(clean_df['label'].value_counts().sort_index().rename(index=LABEL_NAMES).to_string())
clean_df.head()

In [ ]:
# Strict supervised-training validation
invalid_labels = sorted(set(clean_df['label'].unique()) - {0, 1})
if invalid_labels:
    raise ValueError(
        f'Invalid sentiment labels found: {invalid_labels}. Expected only 0 or 1. '
        'Contract: 1 = worried/concerned, 0 = confident/not worried.'
    )

label_counts = clean_df['label'].value_counts().sort_index()
missing_classes = sorted(set([0, 1]) - set(label_counts.index.tolist()))
if missing_classes:
    raise ValueError(
        'Binary sentiment training requires both classes, but these labels are missing: '
        f'{missing_classes}. Current counts: {label_counts.to_dict()}. '
        'Add sentiment=1 examples for worried/concerned posts before training.'
    )

if label_counts.min() < 3:
    raise ValueError(
        f'Each class needs at least 3 rows for stratified train/validation/test splits. '
        f'Current counts: {label_counts.to_dict()}.'
    )

print('Training-data contract passed.')

In [ ]:
# Quick EDA
fig, ax = plt.subplots(figsize=(7, 4))
plot_counts = clean_df['label'].map(LABEL_NAMES).value_counts()
sns.barplot(x=plot_counts.index, y=plot_counts.values, ax=ax)
ax.set_title('Binary sentiment label distribution')
ax.set_xlabel('Label')
ax.set_ylabel('Rows')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

text_lengths = clean_df['text'].str.split().str.len()
print(text_lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
# Stratified train/validation/test split
train_val_df, test_df = train_test_split(
    clean_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=clean_df['label'],
)

relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=relative_val_size,
    random_state=SEED,
    stratify=train_val_df['label'],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for split_name, split_df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(f'\n{split_name}: {len(split_df)} rows')
    print(split_df['label'].value_counts().sort_index().rename(index=LABEL_NAMES).to_string())

In [ ]:
# Load mBERT tokenizer/model from local cache only
try:
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_NAME,
        use_fast=True,
        local_files_only=LOCAL_FILES_ONLY,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        local_files_only=LOCAL_FILES_ONLY,
    )
except OSError as exc:
    raise OSError(
        f'Could not load {BASE_MODEL_NAME} from the local Hugging Face cache. '
        'Download/cache the model once, or set LOCAL_FILES_ONLY=False if network access is allowed.'
    ) from exc

model.config.problem_type = 'single_label_classification'
model.to(DEVICE)

print('Loaded tokenizer:', tokenizer.__class__.__name__)
print('Loaded model:', model.__class__.__name__)
print('Model labels:', model.config.id2label)

In [ ]:
# Tokenized dataset
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['label'].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_ds = SentimentDataset(train_df, tokenizer, MAX_LENGTH)
val_ds = SentimentDataset(val_df, tokenizer, MAX_LENGTH)
test_ds = SentimentDataset(test_df, tokenizer, MAX_LENGTH)

sample = train_ds[0]
print({key: tuple(value.shape) if hasattr(value, 'shape') else value for key, value in sample.items()})

In [ ]:
# Weighted Trainer for imbalanced labels
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


train_counts = train_df['label'].value_counts().sort_index().reindex([0, 1]).astype(float)
class_weights = len(train_df) / (2.0 * train_counts.values)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print('Class weights:', class_weights.tolist())


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    metrics = {
        'accuracy': float(accuracy_score(labels, preds)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
    }
    if len(np.unique(labels)) == 2:
        metrics['roc_auc'] = float(roc_auc_score(labels, probs))
    return metrics

In [ ]:
# mBERT fine-tuning
training_args = TrainingArguments(
    output_dir=str(TRAINER_OUTPUT_DIR),
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    greater_is_better=True,
    report_to=[],
    seed=SEED,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

train_result = trainer.train()
validation_result = trainer.evaluate(eval_dataset=val_ds)
print('Training metrics:')
print(json.dumps(train_result.metrics, indent=2))
print('\nValidation metrics:')
print(json.dumps(validation_result, indent=2))

In [ ]:
# Prediction helpers, threshold tuning, and final test evaluation

def predict_dataset(trainer, dataset):
    prediction_output = trainer.predict(dataset)
    logits = prediction_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    labels = prediction_output.label_ids.astype(int)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    return labels, probs, prediction_output.metrics


def metrics_at_threshold(y_true, worried_probs, threshold):
    y_pred = (worried_probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )
    metrics = {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
    }
    if len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = float(roc_auc_score(y_true, worried_probs))
    else:
        metrics['roc_auc'] = None
    return metrics, y_pred


def tune_threshold(y_true, worried_probs, start=0.05, stop=0.95, step=0.01):
    best = None
    for threshold in np.arange(start, stop + 1e-9, step):
        metrics, _ = metrics_at_threshold(y_true, worried_probs, threshold)
        if best is None or metrics['f1'] > best['f1']:
            best = metrics
    return best


val_labels, val_probs, val_predict_metrics = predict_dataset(trainer, val_ds)
threshold_metrics = tune_threshold(val_labels, val_probs)
selected_threshold = threshold_metrics['threshold']
print('Selected threshold from validation:')
print(json.dumps(threshold_metrics, indent=2))

test_labels, test_probs, test_predict_metrics = predict_dataset(trainer, test_ds)
test_metrics, test_pred_labels = metrics_at_threshold(test_labels, test_probs, selected_threshold)
test_metrics = {
    **test_metrics,
    'trainer_predict_metrics': test_predict_metrics,
}

print('\nTest metrics:')
print(json.dumps(test_metrics, indent=2))
print('\nClassification report:')
print(classification_report(
    test_labels,
    test_pred_labels,
    target_names=[LABEL_NAMES[0], LABEL_NAMES[1]],
    zero_division=0,
))

In [ ]:
# Save artifacts and reports
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

label_mapping_path = ARTIFACT_DIR / 'label_mapping.json'
threshold_path = ARTIFACT_DIR / 'threshold.json'
run_config_path = ARTIFACT_DIR / 'run_config.json'

with open(label_mapping_path, 'w', encoding='utf-8') as f:
    json.dump({str(k): v for k, v in LABEL_NAMES.items()}, f, indent=2)
with open(threshold_path, 'w', encoding='utf-8') as f:
    json.dump({'worried_threshold': selected_threshold}, f, indent=2)
with open(run_config_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model_key': MODEL_KEY,
        'base_model_name': BASE_MODEL_NAME,
        'local_files_only': LOCAL_FILES_ONLY,
        'label_contract': LABEL_NAMES,
        'seed': SEED,
        'training_files': [str(path) for path in TRAINING_FILES],
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
    }, f, indent=2)

metrics_payload = {
    'model_key': MODEL_KEY,
    'base_model_name': BASE_MODEL_NAME,
    'label_contract': LABEL_NAMES,
    'threshold_tuning': threshold_metrics,
    'validation_predict_metrics': val_predict_metrics,
    'test_metrics': test_metrics,
    'class_counts': {
        'train': train_df['label'].value_counts().sort_index().to_dict(),
        'validation': val_df['label'].value_counts().sort_index().to_dict(),
        'test': test_df['label'].value_counts().sort_index().to_dict(),
    },
    'artifacts': {
        'best_model_dir': str(BEST_MODEL_DIR),
        'label_mapping': str(label_mapping_path),
        'threshold': str(threshold_path),
        'run_config': str(run_config_path),
    },
}
with open(REPORT_DIR / 'binary_sentiment_mbert_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)

cm = confusion_matrix(test_labels, test_pred_labels, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=[f'true_{LABEL_NAMES[0]}', f'true_{LABEL_NAMES[1]}'],
    columns=[f'pred_{LABEL_NAMES[0]}', f'pred_{LABEL_NAMES[1]}'],
)
cm_df.to_csv(REPORT_DIR / 'binary_sentiment_mbert_confusion_matrix.csv')

predictions_df = test_df[['text', 'label', 'source_file']].copy()
predictions_df['true_label_name'] = predictions_df['label'].map(LABEL_NAMES)
predictions_df['worried_probability'] = test_probs
predictions_df['predicted_label'] = test_pred_labels
predictions_df['predicted_label_name'] = predictions_df['predicted_label'].map(LABEL_NAMES)
predictions_df['threshold_used'] = selected_threshold
predictions_df.to_csv(REPORT_DIR / 'binary_sentiment_mbert_test_predictions.csv', index=False)

error_df = predictions_df[predictions_df['label'] != predictions_df['predicted_label']].copy()
error_df = error_df.sort_values('worried_probability', ascending=False)
error_df.to_csv(REPORT_DIR / 'binary_sentiment_mbert_error_analysis.csv', index=False)

log_history_df = pd.DataFrame(trainer.state.log_history)
log_history_df.to_csv(REPORT_DIR / 'binary_sentiment_mbert_training_log.csv', index=False)

loss_rows = []
for row in trainer.state.log_history:
    if 'loss' in row:
        loss_rows.append({'epoch': row.get('epoch'), 'split': 'train', 'loss': row['loss']})
    if 'eval_loss' in row:
        loss_rows.append({'epoch': row.get('epoch'), 'split': 'validation', 'loss': row['eval_loss']})
loss_df = pd.DataFrame(loss_rows)
loss_df.to_csv(REPORT_DIR / 'binary_sentiment_mbert_training_validation_loss.csv', index=False)

if not loss_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    for split_name, split_df in loss_df.groupby('split'):
        ax.plot(split_df['epoch'], split_df['loss'], marker='o', label=split_name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('mBERT training and validation loss')
    ax.legend()
    plt.tight_layout()
    fig.savefig(REPORT_DIR / 'binary_sentiment_mbert_training_validation_loss.png', dpi=160)
    plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title('mBERT binary sentiment confusion matrix')
plt.tight_layout()
fig.savefig(REPORT_DIR / 'binary_sentiment_mbert_confusion_matrix.png', dpi=160)
plt.show()

print('Saved artifacts to:', ARTIFACT_DIR)
print('Saved reports to:', REPORT_DIR)

In [ ]:
# Inference helper
best_model = AutoModelForSequenceClassification.from_pretrained(
    BEST_MODEL_DIR,
    local_files_only=True,
).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(
    BEST_MODEL_DIR,
    local_files_only=True,
    use_fast=True,
)
best_model.eval()


def predict_binary_sentiment_mbert(text, threshold=selected_threshold):
    if not isinstance(text, str) or not text.strip():
        raise ValueError('text must be a non-empty string')

    encoded = best_tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
        return_tensors='pt',
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.no_grad():
        logits = best_model(**encoded).logits
        worried_probability = float(torch.softmax(logits, dim=1)[0, 1].detach().cpu())

    predicted_label = int(worried_probability >= threshold)
    return {
        'predicted_label': predicted_label,
        'predicted_label_name': LABEL_NAMES[predicted_label],
        'worried_probability': worried_probability,
        'threshold_used': float(threshold),
    }


examples = [
    'I am worried about my fever and cough getting worse.',
    'I feel okay now and I am not worried about the symptoms.',
]
for text in examples:
    print(text)
    print(json.dumps(predict_binary_sentiment_mbert(text), indent=2))

In [ ]:
# Test cases and scenario checks
# 1) Required training files are intentionally limited to sentiment-labeled files.
assert [path.name for path in TRAINING_FILES] == ['training_1.csv', 'training_2.csv']
assert all(path.exists() for path in TRAINING_FILES)
assert BASE_MODEL_NAME == 'bert-base-multilingual-cased'
assert LOCAL_FILES_ONLY is True

# 2) Normalized data contract.
assert {'text', 'label', 'source_file'}.issubset(clean_df.columns)
assert clean_df['text'].str.len().gt(0).all()
assert set(clean_df['label'].unique()).issubset({0, 1})
assert set(clean_df['label'].unique()) == {0, 1}

# 3) Split reproducibility check.
train_val_df_2, test_df_2 = train_test_split(
    clean_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=clean_df['label'],
)
relative_val_size_2 = VAL_SIZE / (1.0 - TEST_SIZE)
train_df_2, val_df_2 = train_test_split(
    train_val_df_2,
    test_size=relative_val_size_2,
    random_state=SEED,
    stratify=train_val_df_2['label'],
)
assert train_df['text'].tolist() == train_df_2.reset_index(drop=True)['text'].tolist()
assert val_df['text'].tolist() == val_df_2.reset_index(drop=True)['text'].tolist()
assert test_df['text'].tolist() == test_df_2.reset_index(drop=True)['text'].tolist()

# 4) Tokenizer/model smoke check.
smoke_item = test_ds[0]
smoke_inputs = {key: value.unsqueeze(0).to(DEVICE) for key, value in smoke_item.items() if key != 'labels'}
with torch.no_grad():
    smoke_logits = best_model(**smoke_inputs).logits
assert tuple(smoke_logits.shape) == (1, 2)

# 5) Required outputs.
required_paths = [
    BEST_MODEL_DIR / 'config.json',
    ARTIFACT_DIR / 'label_mapping.json',
    ARTIFACT_DIR / 'threshold.json',
    ARTIFACT_DIR / 'run_config.json',
    REPORT_DIR / 'binary_sentiment_mbert_metrics.json',
    REPORT_DIR / 'binary_sentiment_mbert_confusion_matrix.csv',
    REPORT_DIR / 'binary_sentiment_mbert_test_predictions.csv',
    REPORT_DIR / 'binary_sentiment_mbert_error_analysis.csv',
    REPORT_DIR / 'binary_sentiment_mbert_training_log.csv',
    REPORT_DIR / 'binary_sentiment_mbert_training_validation_loss.csv',
    REPORT_DIR / 'binary_sentiment_mbert_confusion_matrix.png',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f'Missing outputs: {missing_paths}'

# 6) Inference schema.
result = predict_binary_sentiment_mbert('I am worried about persistent cough and fever.')
assert set(result.keys()) == {
    'predicted_label',
    'predicted_label_name',
    'worried_probability',
    'threshold_used',
}
assert result['predicted_label'] in {0, 1}
assert result['predicted_label_name'] in set(LABEL_NAMES.values())
assert 0.0 <= result['worried_probability'] <= 1.0

print('All mBERT binary sentiment checks passed.')